# 05 — Area GNN Training & Hierarchical Coordination
Trains GNN forecaster on ward data, runs full hierarchical simulation.
**Run 01, 02, 04 first.**

In [1]:
import sys, os
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path(os.getcwd()).resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')


Project root: D:\Bunker\BaseCamp\Hierarchical-multi-agent-RL-for-Urban-Traffic


## 1. Train Area GNN on collected ward data

In [2]:
import torch
from src.topology import Topology
from src.controllers.area_controller import AreaForecaster

topology = Topology(PROJECT_ROOT)

# --- CONFIGURATION: Select which Area to train the GNN on ---
# Options in your registry: 'HSR_Layout', 'BTM_Layout', 'BTM_Layout_1', 'Jayanagar'
AREA_ID = 'HSR_Layout' 

forecaster = AreaForecaster(AREA_ID, topology)

# Loading the consolidated global RL dataset
gnn_dir = PROJECT_ROOT / 'models' / 'gnn'
global_data_path = gnn_dir / 'global_gnn_data.pt'

if global_data_path.exists():
    all_data = torch.load(global_data_path, weights_only=False)
    print(f'⚡ Loaded {len(all_data)} GNN samples directly from global_gnn_data.pt!')
    
    # Save the combined path for compatibility
    combined_path = gnn_dir / 'combined_training_data.pt'
    torch.save(all_data, combined_path)
    
    print(f'\n🚀 Starting GPU-Powered Offline GNN Training on {forecaster.device}...')
    losses = forecaster.train_offline(combined_path, epochs=100, save_dir=gnn_dir)
    print(f'\n✅ GNN Training Complete! Final MSE Loss: {losses[-1]:.6f}')
else:
    print('❌ No global GNN data found. Please run the fast 20-episode collection run in notebook 04 first!')


❌ No global GNN data found. Please run the fast 20-episode collection run in notebook 04 first!


## 2. Run Full Hierarchical Simulation

In [ ]:
from src.runtime import run_simulation

result = run_simulation(
    scope='ward', identifier='ward_001',
    project_root=PROJECT_ROOT,
    gui=False, scenario_id='normal', max_ticks=600,
)

import json
print(json.dumps(result, indent=2, default=str))

## 3. Results

In [3]:
results_dir = PROJECT_ROOT / 'results' / 'inference'
for f in sorted(results_dir.glob('*.json')):
    print(f'  {f.name}')